# 01 — Data Loading

Studio Proyek 2 · Klasifikasi 5 jenis sampah (IF25-40401)

Tujuan notebook ini: memastikan dataset `train/` terbaca dengan benar dan sesuai format yang diharapkan `evaluate.py`.
Analisis data ada di `02_eda.ipynb`, preprocessing di `03_preprocessing.ipynb`.

## 1. Setup

Tentukan root repo (notebook ada di `notebooks/`, dataset di root), lalu definisikan konstanta yang dipakai di seluruh notebook.
Urutan `CLASSES` harus alfabetis karena itu urutan indeks yang dipakai `ImageFolder` dan `evaluate.py`.

In [13]:
import os
from pathlib import Path
import numpy as np

# Notebook ada di notebooks/, sedangkan train/ dan submission/ ada di root repo
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

DATA_DIR = ROOT / "train"
CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}  # sama dengan evaluate.py

print("root    :", ROOT)
print("dataset :", DATA_DIR, "| ada:", DATA_DIR.is_dir())

root    : d:\deep learning course\stdProyek2_DL_RA\studio-proyek2-cnn-klasifikasi-sampah
dataset : d:\deep learning course\stdProyek2_DL_RA\studio-proyek2-cnn-klasifikasi-sampah\train | ada: True


In [14]:
from collections import Counter

files_by_class, skipped = {}, []
for c in CLASSES:
    files = sorted(p for p in (DATA_DIR / c).iterdir() if p.is_file())
    files_by_class[c] = [p for p in files if p.suffix.lower() in IMG_EXTS]
    skipped += [p for p in files if p.suffix.lower() not in IMG_EXTS]

total = sum(len(fs) for fs in files_by_class.values())
for c, fs in files_by_class.items():
    print(f"{c:<15} {len(fs):>5}")
print(f"{'TOTAL':<15} {total:>5}")

exts = Counter(p.suffix.lower() for fs in files_by_class.values() for p in fs)
print()
print("ekstensi:", dict(exts))
print("file bukan gambar:", [str(p.relative_to(ROOT)) for p in skipped] or "tidak ada")
if set(exts) - {".jpg"}:
    print("PERINGATAN: ada file non-.jpg yang tidak ikut ter-split oleh make_local_split()")

botol_plastik     523
gelas_plastik     521
kertas_kardus     365
saset_kemasan     673
styrofoam         671
TOTAL            2753

ekstensi: {'.jpg': 2753}
file bukan gambar: tidak ada


#### Menambahkan dataset tambahan
tujuan dari penambahan data ini supaya dataset dari setiap kelas menjadi seimbang

In [20]:
ADD_DIR = ROOT / "add_train_balance"

for c in CLASSES:
    d = ADD_DIR / c
    if d.exists():
        extra = sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS)
        files_by_class[c] += extra
        print(f"{c}: +{len(extra)} gambar tambahan")

all_files = [p for fs in files_by_class.values() for p in fs]
labels = np.array([c for c, fs in files_by_class.items() for _ in fs])

# --- tambahan ---
assert len(all_files) == len(labels) == sum(len(fs) for fs in files_by_class.values())

names = Counter((lab, p.name) for p, lab in zip(all_files, labels))
bentrok = [k for k, v in names.items() if v > 1]
print(f"total: {len(all_files)} | nama file bentrok dalam satu kelas: {len(bentrok)}")

botol_plastik: +150 gambar tambahan
gelas_plastik: +152 gambar tambahan
kertas_kardus: +308 gambar tambahan
styrofoam: +2 gambar tambahan
total: 3365 | nama file bentrok dalam satu kelas: 0


Total dari setiap kelas menjadi 673, dengan total 3365

In [17]:
total = sum(len(fs) for fs in files_by_class.values())
for c, fs in files_by_class.items():
    print(f"{c:<15} {len(fs):>5}")
print(f"{'TOTAL':<15} {total:>5}")

botol_plastik     673
gelas_plastik     673
kertas_kardus     673
saset_kemasan     673
styrofoam         673
TOTAL            3365


## 2. Cek struktur folder kelas

Harus ada tepat 5 folder kelas di `train/`, dengan nama yang sama persis dengan `CLASSES`.

In [18]:
assert DATA_DIR.is_dir(), f"Folder {DATA_DIR} tidak ditemukan. Ekstrak dataset ke train/<nama_kelas>/..."

found = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
assert found == CLASSES, f"Folder kelas tidak sesuai. diharapkan: {CLASSES}, ditemukan: {found}"

for i, c in enumerate(CLASSES):
    print(f"{i} -> {c}")

0 -> botol_plastik
1 -> gelas_plastik
2 -> kertas_kardus
3 -> saset_kemasan
4 -> styrofoam


## 3. Cek gambar bisa dibuka

Buka setiap gambar dengan PIL untuk memastikan tidak ada file rusak sebelum masuk ke EDA dan preprocessing.

In [21]:
from PIL import Image

corrupt = []
for p in all_files:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        corrupt.append((p, repr(e)))

print(f"terbaca: {len(all_files) - len(corrupt)} / {len(all_files)}")
for p, err in corrupt:
    print("  rusak:", p.relative_to(ROOT), "->", err)

terbaca: 3365 / 3365


## 5. Ringkasan

Dataset siap dianalisis di `02_eda.ipynb` bila:
- 5 folder kelas sesuai `CLASSES`,
- tidak ada file non-gambar atau non-`.jpg` (atau sudah ditangani),
- tidak ada gambar rusak.